In [11]:
# Summarize ILC sampels (n=112)
import csv
import re
from collections import defaultdict
from pathlib import Path

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side


# ==============================================================
# Paths
# ==============================================================

RESULTS_DIR = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/"
    "APOLLO_Broad/ILC/ichorCNA/hg38_1000kb/results/ichorCNA"
)

CLINICAL_XLSX = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/"
    "Clinical_data/deidentified APOLLO 09-04-26.xlsx"
)

OUTPUT_CSV = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/ILC/ichorCNA/hg38_1000kb/"
    "ichorCNA_ILC_summary.csv"
)

OUTPUT_XLSX = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/"
    "APOLLO_Broad/ILC/"
    "ichorCNA_ILC_summary_grouped.xlsx"
)


# ==============================================================
# Output columns
# ==============================================================

OUTPUT_COLUMNS = [
    "Bios#",
    "Redcap ID",
    "Rapid Autopsy status",
    "Histology IDC/ILC",
    "Sample ID (TP number)",
    "Seq Status",
    "Treatment Line",
    "ER Status",
    "HER-2 Status",
    "Metastatic_sites",
    "Tumor_fraction",
    "Ploidy",
    "Subclol_fraction",
    "Frac_Genome_Subclol",
    "Frac_C_Subclol",
]

METRIC_COLUMNS = [
    "Tumor_fraction",
    "Ploidy",
    "Subclol_fraction",
    "Frac_Genome_Subclol",
    "Frac_C_Subclol",
]

TP_PATTERN = re.compile(r"TP\s*(\d+)\s*-\s*M\s*(\d+)", re.IGNORECASE)


# ==============================================================
# Helper functions
# ==============================================================

def clean_text(value):
    """Convert any Excel value to clean text."""
    if value is None:
        return ""
    return str(value).strip()


def normalise_tp(value):
    """Extract and standardize a TP ID, e.g. TP22-M104."""
    match = TP_PATTERN.search(clean_text(value))
    if not match:
        return None
    return f"TP{int(match.group(1)):02d}-M{int(match.group(2))}"


def normalise_bios(value):
    """Standardize Bios# so 37939.0 becomes 37939."""
    value = clean_text(value)

    if not value:
        return ""

    try:
        return str(int(float(value)))
    except ValueError:
        return value


def numeric_or_blank(value):
    """Write numerical iChorCNA metrics as numeric Excel values."""
    value = clean_text(value)

    if not value or value.upper() == "NA":
        return ""

    try:
        return float(value)
    except ValueError:
        return value


def normalise_header(value):
    """Normalize clinical headers for flexible matching."""
    return re.sub(r"[^a-z0-9]+", "", clean_text(value).lower())


def find_column(headers, accepted_names):
    """
    Find a column using flexible header names.
    Example: BIOS #, BIOS#, or Bios# are treated as equivalent.
    """
    normalized_headers = [normalise_header(x) for x in headers]
    accepted_normalized = [normalise_header(x) for x in accepted_names]

    for accepted in accepted_normalized:
        if accepted in normalized_headers:
            return normalized_headers.index(accepted)

    return None


def extract_ichorcna_metrics(params_file):
    """Read tumor fraction, ploidy and subclonal metrics from params file."""

    label_map = {
        "Tumor Fraction": "Tumor_fraction",
        "Ploidy": "Ploidy",
        "Subclone Fraction": "Subclol_fraction",
        "Fraction Genome Subclonal": "Frac_Genome_Subclol",
        "Fraction CNA Subclonal": "Frac_C_Subclol",
    }

    metrics = {column: "" for column in METRIC_COLUMNS}

    for line in params_file.read_text(errors="replace").splitlines():
        for label, output_column in label_map.items():
            match = re.match(
                rf"^\s*{re.escape(label)}\s*:\s*(.+?)\s*$",
                line,
            )

            if match:
                value = match.group(1).strip()
                metrics[output_column] = "" if value.upper() == "NA" else value

    return metrics


# ==============================================================
# Read clinical workbook
# ==============================================================

if not CLINICAL_XLSX.is_file():
    raise FileNotFoundError(f"Clinical workbook not found:\n{CLINICAL_XLSX}")

clinical_wb = load_workbook(
    CLINICAL_XLSX,
    read_only=True,
    data_only=True,
)

# TP → complete clinical record
clinical_lookup = {}

# TP → Bios# for corrections
clinical_tp_to_bios = {}

# Bios# → TP sample sequence from clinical sheet
clinical_tp_order = defaultdict(dict)

# Bios# → clinical case order
clinical_bios_order = {}

# QC
tp_conflicts = []
bios_counter = 0

# Higher score = preferred clinical record if the same TP appears in multiple sheets.
sheet_settings = [
    ("AUTOPSY PATIENTS", 4),
    ("ONGOING COLLECTIONS", 3),
    ("used plasma", 2),
    ("SERIAL COLLECTIONS", 1),
]

record_priority = {}


def register_clinical_record(
    sample_id,
    record,
    bios,
    sequence_number,
    sheet_name,
    row_number,
    priority,
    source,
):
    """Add a TP-to-clinical match, while recording conflicts."""

    if not sample_id or not bios:
        return

    # TP-to-Bios# conflict check
    if sample_id in clinical_tp_to_bios:
        previous_bios = clinical_tp_to_bios[sample_id]

        if previous_bios != bios:
            tp_conflicts.append(
                {
                    "TP": sample_id,
                    "previous_Bios#": previous_bios,
                    "new_Bios#": bios,
                    "sheet": sheet_name,
                    "row": row_number,
                    "source": source,
                }
            )

    # Use the highest-priority record.
    if (
        sample_id not in clinical_lookup
        or priority > record_priority[sample_id]
    ):
        clinical_lookup[sample_id] = record
        clinical_tp_to_bios[sample_id] = bios
        record_priority[sample_id] = priority

    # Preserve longitudinal TP order within the case.
    if sample_id not in clinical_tp_order[bios]:
        clinical_tp_order[bios][sample_id] = sequence_number


for sheet_name, priority in sheet_settings:
    if sheet_name not in clinical_wb.sheetnames:
        continue

    ws = clinical_wb[sheet_name]

    # Find the true header row.
    # ONGOING COLLECTIONS has a note row before its headers.
    header_row_number = None
    headers = None

    for row_number in range(1, min(ws.max_row, 10) + 1):
        row_values = [
            ws.cell(row_number, column).value
            for column in range(1, ws.max_column + 1)
        ]

        clean_headers = [clean_text(x) for x in row_values]

        bios_test = find_column(clean_headers, ["BIOS #", "BIOS#"])
        tp_test = [
            index
            for index, value in enumerate(clean_headers)
            if normalise_header(value) == "tpnumber"
        ]

        if bios_test is not None and tp_test:
            header_row_number = row_number
            headers = clean_headers
            break

    if header_row_number is None:
        print(f"Warning: header row not found in {sheet_name}")
        continue

    # Identify main patient-level columns.
    bios_idx = find_column(headers, ["BIOS #", "BIOS#"])
    redcap_idx = find_column(headers, ["REDCAP#", "REDCAP #"])
    histology_idx = find_column(headers, ["IDC/ILC"])
    er_idx = find_column(headers, ["ER Status"])
    her2_idx = find_column(
        headers,
        ["HER2 Status", "HER-2 Status", "HER2"],
    )
    name_idx = find_column(headers, ["Name"])

    tp_indices = [
        index
        for index, value in enumerate(headers)
        if normalise_header(value) == "tpnumber"
    ]

    tissue_indices = [
        index
        for index, value in enumerate(headers)
        if normalise_header(value) == "tissueprogressedto"
    ]

    for row_number in range(header_row_number + 1, ws.max_row + 1):
        row = [
            ws.cell(row_number, column).value
            for column in range(1, ws.max_column + 1)
        ]

        bios = normalise_bios(row[bios_idx] if bios_idx is not None else "")

        if not bios:
            continue

        if bios not in clinical_bios_order:
            clinical_bios_order[bios] = bios_counter
            bios_counter += 1

        redcap = clean_text(row[redcap_idx]) if redcap_idx is not None else ""
        histology = clean_text(row[histology_idx]) if histology_idx is not None else ""
        er_status = clean_text(row[er_idx]) if er_idx is not None else ""
        her2_status = clean_text(row[her2_idx]) if her2_idx is not None else ""

        rapid_autopsy_status = ""
        if sheet_name == "AUTOPSY PATIENTS" and name_idx is not None:
            rapid_autopsy_status = clean_text(row[name_idx])

        # ----------------------------------------------------------
        # Regular TP Number columns, left-to-right clinical sequence
        # ----------------------------------------------------------
        sequence_number = 0

        for tp_idx in tp_indices:
            raw_tp = clean_text(row[tp_idx])
            sample_id = normalise_tp(raw_tp)

            if not sample_id:
                continue

            # Find nearest "Tissue Progressed to" column to the left.
            possible_tissue_cols = [
                tissue_idx
                for tissue_idx in tissue_indices
                if tissue_idx < tp_idx
            ]

            tissue_site = ""
            if possible_tissue_cols:
                nearest_tissue_idx = max(possible_tissue_cols)
                tissue_site = clean_text(row[nearest_tissue_idx])

            record = {
                "Bios#": bios,
                "Redcap ID": redcap,
                "Rapid Autopsy status": rapid_autopsy_status,
                "Histology IDC/ILC": histology,
                "Sample ID (TP number)": raw_tp,
                "Seq Status": "Done",
                "Treatment Line": "",
                "ER Status": er_status,
                "HER-2 Status": her2_status,
                "Metastatic_sites": tissue_site,
            }

            register_clinical_record(
                sample_id=sample_id,
                record=record,
                bios=bios,
                sequence_number=sequence_number,
                sheet_name=sheet_name,
                row_number=row_number,
                priority=priority,
                source="TP Number column",
            )

            sequence_number += 1

        # ----------------------------------------------------------
        # Autopsy Name field:
        # Example: RA#93 TP25-M5
        # ----------------------------------------------------------
        if sheet_name == "AUTOPSY PATIENTS" and name_idx is not None:
            raw_name = clean_text(row[name_idx])
            name_tp = normalise_tp(raw_name)

            if name_tp:
                record = {
                    "Bios#": bios,
                    "Redcap ID": redcap,
                    "Rapid Autopsy status": raw_name,
                    "Histology IDC/ILC": histology,
                    "Sample ID (TP number)": name_tp,
                    "Seq Status": "Done",
                    "Treatment Line": "",
                    "ER Status": er_status,
                    "HER-2 Status": her2_status,
                    "Metastatic_sites": "",
                }

                register_clinical_record(
                    sample_id=name_tp,
                    record=record,
                    bios=bios,
                    sequence_number=sequence_number,
                    sheet_name=sheet_name,
                    row_number=row_number,
                    priority=priority,
                    source="AUTOPSY PATIENTS Name column",
                )


# ==============================================================
# Parse iChorCNA params files and create summary rows
# ==============================================================

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f"Results directory not found:\n{RESULTS_DIR}")

params_files = sorted(RESULTS_DIR.rglob("*.params.txt"))

summary_rows = []
unmatched_samples = []
unreadable_files = []

for params_file in params_files:
    sample_id = normalise_tp(params_file.stem.replace(".params", ""))

    # Backup: use folder name if required.
    if not sample_id:
        sample_id = normalise_tp(params_file.parent.name)

    if not sample_id:
        unreadable_files.append(str(params_file))
        continue

    row = {column: "" for column in OUTPUT_COLUMNS}

    if sample_id in clinical_lookup:
        row.update(clinical_lookup[sample_id])
    else:
        row["Sample ID (TP number)"] = sample_id
        row["Seq Status"] = "Done"
        unmatched_samples.append(sample_id)

    row.update(extract_ichorcna_metrics(params_file))
    row["_tp"] = sample_id
    row["_bios"] = normalise_bios(row["Bios#"])
    summary_rows.append(row)


# ==============================================================
# Order cases and samples
# ==============================================================

first_csv_position = {}

for index, row in enumerate(summary_rows):
    row["_source_order"] = index

    bios = row["_bios"]

    if bios and bios not in first_csv_position:
        first_csv_position[bios] = index


def bios_sort_key(row):
    bios = row["_bios"]

    if bios:
        return (
            0,
            clinical_bios_order.get(bios, 10**9),
            first_csv_position.get(bios, row["_source_order"]),
        )

    # Unmatched samples are placed at the end.
    return (1, row["_source_order"], 0)


summary_rows.sort(key=bios_sort_key)

ordered_rows = []
bios_groups = []

for row in summary_rows:
    bios = row["_bios"]

    if bios not in bios_groups:
        bios_groups.append(bios)

for bios in bios_groups:
    group = [row for row in summary_rows if row["_bios"] == bios]

    group.sort(
        key=lambda row: (
            clinical_tp_order.get(bios, {}).get(row["_tp"], 10**9),
            row["_source_order"],
        )
    )

    ordered_rows.extend(group)


# ==============================================================
# Write corrected CSV
# ==============================================================

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()

    for row in ordered_rows:
        writer.writerow(
            {column: row.get(column, "") for column in OUTPUT_COLUMNS}
        )


# ==============================================================
# Create grouped Excel workbook
# ==============================================================

wb = Workbook()
ws = wb.active
ws.title = "ichorCNA_summary"
ws.sheet_view.showGridLines = False
ws.freeze_panes = "A2"

# Styling
header_fill = PatternFill("solid", fgColor="1F4E78")
bios_fill = PatternFill("solid", fgColor="EAF2F8")

header_font = Font(bold=True, color="FFFFFF", size=11)
body_font = Font(size=10)
bios_font = Font(bold=True, size=10)

thin_blue = Side(style="thin", color="9FBAD0")
thin_light = Side(style="thin", color="D9E2F3")
medium_blue = Side(style="medium", color="9FBAD0")

# Header
for col_index, header in enumerate(OUTPUT_COLUMNS, start=1):
    cell = ws.cell(1, col_index, header)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )
    cell.border = Border(
        left=thin_blue,
        right=thin_blue,
        top=thin_blue,
        bottom=thin_blue,
    )

ws.row_dimensions[1].height = 34

# Data
for excel_row, row in enumerate(ordered_rows, start=2):
    for col_index, header in enumerate(OUTPUT_COLUMNS, start=1):
        value = row.get(header, "")

        if header in METRIC_COLUMNS:
            value = numeric_or_blank(value)

        cell = ws.cell(excel_row, col_index, value)
        cell.font = body_font
        cell.border = Border(bottom=thin_light)
        cell.alignment = Alignment(vertical="center")

        if header in METRIC_COLUMNS:
            cell.number_format = "0.0000"
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
            )

        elif header in {
            "Bios#",
            "Redcap ID",
            "Rapid Autopsy status",
            "Histology IDC/ILC",
            "Seq Status",
            "Treatment Line",
            "ER Status",
            "HER-2 Status",
        }:
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
            )

        elif header == "Metastatic_sites":
            cell.alignment = Alignment(
                vertical="center",
                wrap_text=True,
            )

    ws.row_dimensions[excel_row].height = 20


# ==============================================================
# Merge Bios# cells within each case
# ==============================================================

bios_column = OUTPUT_COLUMNS.index("Bios#") + 1

start_index = 0

while start_index < len(ordered_rows):
    bios = ordered_rows[start_index]["_bios"]
    end_index = start_index + 1

    while (
        end_index < len(ordered_rows)
        and ordered_rows[end_index]["_bios"] == bios
    ):
        end_index += 1

    excel_start = start_index + 2
    excel_end = end_index + 1

    # Merge only known Bios# groups with ≥2 samples.
    if bios and (end_index - start_index) > 1:
        ws.merge_cells(
            start_row=excel_start,
            start_column=bios_column,
            end_row=excel_end,
            end_column=bios_column,
        )

        merged_cell = ws.cell(excel_start, bios_column)
        merged_cell.value = bios
        merged_cell.fill = bios_fill
        merged_cell.font = bios_font
        merged_cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
        )
        merged_cell.border = Border(
            left=thin_blue,
            right=thin_blue,
            top=thin_blue,
            bottom=thin_blue,
        )

    # Stronger top border for each Bios# group.
    if bios:
        for column in range(1, len(OUTPUT_COLUMNS) + 1):
            ws.cell(excel_start, column).border = Border(
                top=medium_blue,
                bottom=thin_light,
            )

    start_index = end_index


# ==============================================================
# Column widths
# ==============================================================

column_widths = {
    "A": 11,   # Bios#
    "B": 11,   # Redcap ID
    "C": 20,   # Rapid Autopsy status
    "D": 18,   # Histology
    "E": 22,   # Sample ID
    "F": 12,   # Seq Status
    "G": 20,   # Treatment Line
    "H": 13,   # ER Status
    "I": 13,   # HER-2 Status
    "J": 30,   # Metastatic_sites
    "K": 17,   # Tumor_fraction
    "L": 14,   # Ploidy
    "M": 18,   # Subclol_fraction
    "N": 22,   # Frac_Genome_Subclol
    "O": 18,   # Frac_C_Subclol
}

for column_letter, width in column_widths.items():
    ws.column_dimensions[column_letter].width = width


# ==============================================================
# Save outputs and QC summary
# ==============================================================

OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
wb.save(OUTPUT_XLSX)

matched_clinical = sum(
    row["_tp"] in clinical_lookup
    for row in ordered_rows
)

print("=" * 70)
print("iChorCNA ILC summary completed")
print("=" * 70)
print(f"Params files found:           {len(params_files)}")
print(f"Rows written:                 {len(ordered_rows)}")
print(f"Clinical TP matches:          {matched_clinical}")
print(f"Clinical TP unmatched:        {len(unmatched_samples)}")
print(f"Unique Bios# groups:          {len([x for x in bios_groups if x])}")
print(f"Output CSV:                   {OUTPUT_CSV}")
print(f"Output Excel workbook:        {OUTPUT_XLSX}")

if unmatched_samples:
    print("\nUnmatched iChorCNA samples:")
    print(", ".join(sorted(unmatched_samples)))

if unreadable_files:
    print("\nFiles without recognizable TP sample IDs:")
    print("\n".join(unreadable_files))

if tp_conflicts:
    print("\nClinical TP-to-Bios# conflicts detected:")
    for conflict in tp_conflicts:
        print(conflict)

iChorCNA ILC summary completed
Params files found:           112
Rows written:                 112
Clinical TP matches:          112
Clinical TP unmatched:        0
Unique Bios# groups:          32
Output CSV:                   /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/ILC/ichorCNA/hg38_1000kb/ichorCNA_ILC_summary.csv
Output Excel workbook:        /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/ILC/ichorCNA_ILC_summary_grouped.xlsx

Clinical TP-to-Bios# conflicts detected:
{'TP': 'TP23-M286', 'previous_Bios#': '85443', 'new_Bios#': '93957', 'sheet': 'ONGOING COLLECTIONS', 'row': 48, 'source': 'TP Number column'}
{'TP': 'TP23-M266', 'previous_Bios#': '77542', 'new_Bios#': '78822', 'sheet': 'ONGOING COLLECTIONS', 'row': 97, 'source': 'TP Number column'}
{'TP': 'TP22-M138', 'previous_Bios#': '54478', 'new_Bios#': '70555', 'sheet': 'ONGOING COLLECTIONS', 'row': 161, 'source': 'TP Number column'}
{'TP': 'TP24-M24', 'previous_Bios#': '94430', 'new_Bi

In [1]:
import csv
import re
import sys
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
RESULTS_DIR = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/"
    "APOLLO_Broad/20260902_ctDNA_shWGS/ichorCNA/results/ichorCNA"
)

CLINICAL_XLSX = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/"
    "Clinical_data/deidentified APOLLO 09-04-26.xlsx"
)

OUTPUT_CSV = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/"
    "APOLLO_Broad/20260902_ctDNA_shWGS/ichorCNA/"
    "ichorCNA_summary.csv"
)

# ------------------------------------------------------------------
# Output columns
# ------------------------------------------------------------------
OUTPUT_COLUMNS = [
    "Bios#",
    "Redcap ID",
    "Rapid Autopsy status",
    "Histology IDC/ILC",
    "Sample ID (TP number)",
    "Seq Status",
    "Treatment Line",
    "ER Status",
    "HER-2 Status",
    "Metastatic_sites",
    "Tumor_fraction",
    "Ploidy",
    "Subclol_fraction",
    "Frac_Genome_Subclol",
    "Frac_C_Subclol",
]

TP_PATTERN = re.compile(r"TP\d{2}-M\d+", re.IGNORECASE)

XML_NS = {
    "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "pkg": "http://schemas.openxmlformats.org/package/2006/relationships",
}


def clean_text(value):
    """Convert Excel/XML values to clean text."""
    if value is None:
        return ""
    return str(value).strip()


def canonical_tp(value):
    """Extract canonical TP ID, e.g., TP17-M581."""
    match = TP_PATTERN.search(clean_text(value).upper())
    return match.group(0).upper() if match else None


def excel_column_index(cell_reference):
    """Convert Excel reference, e.g. AB12, into zero-based column index."""
    letters = re.match(r"[A-Z]+", cell_reference).group(0)

    index = 0
    for letter in letters:
        index = index * 26 + ord(letter) - ord("A") + 1

    return index - 1


def read_xlsx_sheet(workbook_path, requested_sheet):
    """
    Read one Excel sheet using only Python standard-library modules.
    This avoids pandas/openpyxl dependency problems.
    """
    with zipfile.ZipFile(workbook_path) as archive:

        # Read shared strings, if present
        shared_strings = []
        if "xl/sharedStrings.xml" in archive.namelist():
            shared_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))

            for string_item in shared_root.findall("main:si", XML_NS):
                shared_strings.append("".join(string_item.itertext()))

        # Find requested sheet inside workbook
        workbook_xml = ET.fromstring(archive.read("xl/workbook.xml"))
        rels_xml = ET.fromstring(archive.read("xl/_rels/workbook.xml.rels"))

        relationship_targets = {
            relationship.attrib["Id"]: relationship.attrib["Target"]
            for relationship in rels_xml.findall("pkg:Relationship", XML_NS)
        }

        sheet_target = None

        for sheet in workbook_xml.findall("main:sheets/main:sheet", XML_NS):
            if sheet.attrib.get("name") == requested_sheet:
                relationship_id = sheet.attrib[
                    "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
                ]
                sheet_target = relationship_targets[relationship_id]
                break

        if sheet_target is None:
            raise ValueError(f"Sheet not found: {requested_sheet}")

        sheet_path = "xl/" + sheet_target.lstrip("/")
        sheet_xml = ET.fromstring(archive.read(sheet_path))

    all_rows = []

    for row_xml in sheet_xml.findall("main:sheetData/main:row", XML_NS):
        row_values = []

        for cell in row_xml.findall("main:c", XML_NS):
            column_index = excel_column_index(cell.attrib["r"])

            while len(row_values) <= column_index:
                row_values.append("")

            cell_type = cell.attrib.get("t")
            value_xml = cell.find("main:v", XML_NS)

            if cell_type == "s" and value_xml is not None:
                row_values[column_index] = shared_strings[int(value_xml.text)]

            elif cell_type == "inlineStr":
                row_values[column_index] = "".join(cell.itertext())

            elif value_xml is not None:
                row_values[column_index] = value_xml.text or ""

        all_rows.append(row_values)

    return all_rows


def extract_ichorcna_metrics(params_file):
    """Extract iChorCNA metrics from one *.params.txt file."""

    labels = {
        "Tumor Fraction": "Tumor_fraction",
        "Ploidy": "Ploidy",
        "Subclone Fraction": "Subclol_fraction",
        "Fraction Genome Subclonal": "Frac_Genome_Subclol",
        "Fraction CNA Subclonal": "Frac_C_Subclol",
    }

    metrics = {column: "" for column in labels.values()}

    for line in params_file.read_text(errors="replace").splitlines():
        for label, output_column in labels.items():
            match = re.match(
                rf"^\s*{re.escape(label)}\s*:\s*(.+?)\s*$",
                line
            )

            if match:
                value = match.group(1).strip()
                metrics[output_column] = "" if value.upper() == "NA" else value

    return metrics


def clinical_records_from_sheet(workbook_path, sheet_name, header_row, is_autopsy):
    """
    Build TP-level clinical records.

    Each TP Number column is paired with its Tissue Progressed to column,
    located three columns to the left.
    """

    raw = read_xlsx_sheet(workbook_path, sheet_name)

    headers = [clean_text(x) for x in raw[header_row]]
    data_rows = raw[header_row + 1:]

    def header_index(label):
        if label not in headers:
            raise ValueError(f"Column '{label}' not found in sheet '{sheet_name}'")
        return headers.index(label)

    histology_idx = header_index("IDC/ILC")
    redcap_idx = header_index("REDCAP#")
    bios_idx = header_index("BIOS #")
    er_idx = header_index("ER Status")

    if is_autopsy:
        her2_idx = header_index("HER2 Status")
        name_idx = header_index("Name")
    else:
        her2_idx = header_index("HER-2 Status")
        name_idx = None

    tp_indices = [
        index for index, header in enumerate(headers)
        if header == "TP Number"
    ]

    for row in data_rows:

        # Extend short rows to header width
        row = row + [""] * max(0, len(headers) - len(row))

        for tp_idx in tp_indices:
            tp_cell = clean_text(row[tp_idx])
            sample_id = canonical_tp(tp_cell)

            if not sample_id:
                continue

            tissue_site = clean_text(row[tp_idx - 3])

            yield sample_id, {
                "Bios#": clean_text(row[bios_idx]),
                "Redcap ID": clean_text(row[redcap_idx]),
                "Rapid Autopsy status": (
                    clean_text(row[name_idx]) if is_autopsy else ""
                ),
                "Histology IDC/ILC": clean_text(row[histology_idx]),
                "Sample ID (TP number)": tp_cell,
                "Seq Status": "Done",
                "Treatment Line": "",
                "ER Status": clean_text(row[er_idx]),
                "HER-2 Status": clean_text(row[her2_idx]),
                "Metastatic_sites": tissue_site,
            }


# ------------------------------------------------------------------
# Build TP-to-clinical-data lookup
# AUTOPSY PATIENTS takes priority because it contains Rapid Autopsy status
# ------------------------------------------------------------------
clinical_lookup = {}
duplicate_clinical_tp_count = 0

sheet_settings = [
    ("ONGOING COLLECTIONS", 1, False),
    ("AUTOPSY PATIENTS", 0, True),
]

for sheet_name, header_row, is_autopsy in sheet_settings:

    for sample_id, record in clinical_records_from_sheet(
        CLINICAL_XLSX,
        sheet_name,
        header_row,
        is_autopsy
    ):

        if sample_id in clinical_lookup:
            duplicate_clinical_tp_count += 1

            # Replace with AUTOPSY PATIENTS record when applicable
            if is_autopsy:
                clinical_lookup[sample_id] = record

        else:
            clinical_lookup[sample_id] = record


# ------------------------------------------------------------------
# Parse all iChorCNA parameter files and create output rows
# ------------------------------------------------------------------
if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f"Results directory not found:\n{RESULTS_DIR}")

if not CLINICAL_XLSX.is_file():
    raise FileNotFoundError(f"Clinical workbook not found:\n{CLINICAL_XLSX}")

params_files = sorted(RESULTS_DIR.rglob("*.params.txt"))

summary_rows = []
unmatched_samples = []
unreadable_sample_ids = []

for params_file in params_files:

    # Usually extracted from: TP17-M581.params.txt
    sample_id = canonical_tp(params_file.stem.replace(".params", ""))

    # Backup: extract from parent folder name
    if not sample_id:
        sample_id = canonical_tp(params_file.parent.name)

    if not sample_id:
        unreadable_sample_ids.append(str(params_file))
        continue

    row = {column: "" for column in OUTPUT_COLUMNS}

    if sample_id in clinical_lookup:
        row.update(clinical_lookup[sample_id])

    else:
        row["Sample ID (TP number)"] = sample_id
        row["Seq Status"] = "Done"
        unmatched_samples.append(sample_id)

    row.update(extract_ichorcna_metrics(params_file))
    summary_rows.append(row)


# ------------------------------------------------------------------
# Write CSV
# ------------------------------------------------------------------
summary_rows.sort(
    key=lambda x: canonical_tp(x["Sample ID (TP number)"]) or ""
)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as output_handle:
    writer = csv.DictWriter(output_handle, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()
    writer.writerows(summary_rows)


# ------------------------------------------------------------------
# QC summary
# ------------------------------------------------------------------
with_tumor_fraction = sum(
    bool(row["Tumor_fraction"]) for row in summary_rows
)

matched_clinical = len(summary_rows) - len(unmatched_samples)

print("=" * 65)
print("iChorCNA summary completed")
print("=" * 65)
print(f"Output CSV:                    {OUTPUT_CSV}")
print(f"*.params.txt files found:      {len(params_files)}")
print(f"Rows written:                  {len(summary_rows)}")
print(f"Rows with tumor fraction:      {with_tumor_fraction}")
print(f"Clinical TP IDs matched:       {matched_clinical}")
print(f"Clinical TP IDs unmatched:     {len(unmatched_samples)}")
print(f"Duplicate clinical TP records: {duplicate_clinical_tp_count}")
print(f"Unreadable sample IDs:         {len(unreadable_sample_ids)}")

if unmatched_samples:
    print("\nUnmatched samples:")
    print(", ".join(sorted(unmatched_samples)))

if unreadable_sample_ids:
    print("\nFiles without a recognizable TP ID:")
    print("\n".join(unreadable_sample_ids))

print("\nFirst five output rows:")

for row in summary_rows[:5]:
    print(
        row["Sample ID (TP number)"],
        "| TFx =", row["Tumor_fraction"],
        "| Ploidy =", row["Ploidy"],
        "| Histology =", row["Histology IDC/ILC"],
        "| RA =", row["Rapid Autopsy status"]
    )

iChorCNA summary completed
Output CSV:                    /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/APOLLO/APOLLO_Broad/20260902_ctDNA_shWGS/ichorCNA/ichorCNA_summary.csv
*.params.txt files found:      438
Rows written:                  438
Rows with tumor fraction:      438
Clinical TP IDs matched:       438
Clinical TP IDs unmatched:     0
Duplicate clinical TP records: 4
Unreadable sample IDs:         0

First five output rows:
TP17-M581(4) | TFx = 0.0745 | Ploidy = 2.888 | Histology = IDC | RA = RA#5
TP17-M746(11) | TFx = 0.04566 | Ploidy = 1.996 | Histology = IDC | RA = RA#3-TP19-M307
TP17-M786(8) | TFx = 0.06364 | Ploidy = 1.752 | Histology = IDC | RA = RA#5
TP18-M281 (3) | TFx = 0.3743 | Ploidy = 2.288 | Histology = IDC | RA = RA#8-TP20-M204
TP18-M315(7) | TFx = 0 | Ploidy = 1.999 | Histology = IDC | RA = RA#13-TP18-M299
